## Analysis Notebook Start

Section Content
- Package imports
- Data imports
- Data cleaning & normalization
- Set common variables

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import datetime
import plotly.graph_objects as go


load_dotenv()

# Common Variables
today = datetime.date.today()
cutoff20 = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3

ticker = 'TXN'
audit = True

path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)

# Audit Export
path_analysis_csv = os.path.join(path_stockdata, f'{ticker.upper()}--Analysis=DivBond-s1v1.csv')

# Data Import
dp_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--DivPrice_History-s1v1.csv'), index_col=0)
dp_aggr_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Aggregate_Cy_DivPrice_History-s1v1.csv'), index_col=0)
ann10k_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Annual_10K-s1v1.csv'), index_col=0)


In [ ]:
%%capture
# Data Cleaning Div Price
dp_df0['Date'] = pd.to_datetime(dp_df0['Date'])
dp_df1 = dp_df0[dp_df0['Date'].dt.year >= cutoff20]
dp_df1['FwdDiv%'] = dp_df0['FwdDivYield']


# Data Cleaning Aggregate Div Price
dp_aggr_df1 = dp_aggr_df0.query('DateCy >= @cutoff20 and DateCy < @today.year')

# Data Cleaning Annual 10K Data
ann10k_df0 = ann10k_df0.dropna(subset=['FiscalYear'])
ann10k_df0[['FiscalYear', 'FiscalMonth']] = ann10k_df0[['FiscalYear', 'FiscalMonth']].astype(int)
ann10k_df1 = ann10k_df0[ann10k_df0['FiscalYear'] >= cutoff20]

# Margins
margin_df = ann10k_df1[['FiscalYear', 'Revenue', 'GrossProfit', 'OperatingIncome', 'NetIncome', 'OpCash', 'FreeCash', 'CAPEX']]
margin_df['GPM'] = margin_df['GrossProfit'] / margin_df['Revenue']
margin_df['OPM'] = margin_df['OperatingIncome'] / margin_df['Revenue']
margin_df['NPM'] = margin_df['NetIncome'] / margin_df['Revenue']
margin_df['OCM'] = margin_df['OpCash'] / margin_df['Revenue']
margin_df['FCM'] = margin_df['FreeCash'] / margin_df['Revenue']
margin_df['CXM'] = margin_df['CAPEX'] / margin_df['Revenue']

# DvD
dvd_df = ann10k_df1[['FiscalYear', 'Revenue', 'OpCash', 'FreeCash', 'CAPEX', 'DivCash', 'StockIssue', 'StockBuyBack',
                     'C&E', 'TreasuryStock',  'CurrentAssets', 'TotalAssets', 'ShortDebt', 'LongDebt', 'CurrentLiabilities', 'TotalLiabilities']]

# GRO
gro_df0 = ann10k_df0[['FiscalYear', 'Revenue', 'OpCash', 'DivCash', 'RevPS', 'OpCashPS', 'DivPS']]
gro_df0['RevGro'] = round(gro_df0['Revenue'].pct_change() , 2)
gro_df0['OpCashGro'] = round(gro_df0['Revenue'].pct_change() , 2)
gro_df0['DivGro'] = round(gro_df0['DivCash'].pct_change() , 2)
gro_df0['RevGroPS'] = round(gro_df0['RevPS'].pct_change() , 2)
gro_df0['OpCashGroPS'] = round(gro_df0['OpCashPS'].pct_change() , 2)
gro_df0['DivGroPS'] = round(gro_df0['DivPS'].pct_change() , 2)
gro_df1 = gro_df0.tail(20)


In [ ]:
#dp_df1

In [ ]:
#dp_aggr_df1

In [ ]:
#ann10k_df1

In [ ]:
#margin_df

In [ ]:
#dvd_df

In [ ]:
#gro_df1

## Dividend Value Theory
Steps
1. View aggregate dividend yields for last 10 years
2. View Mean & Medians (M&Ms) for last 20 years
3. Make initial gate prediction
4. Analysis of DYT chart with gates
5. Confirm gate values

In [ ]:
dp_aggr_df1[['DivYieldMin', 'DivYieldMax', 'DivYieldMean', 'DivYieldMedian']].style.format({'DivYieldMin': '{:.2%}', 'DivYieldMax': '{:.2%}',
                                                                                           'DivYieldMean': '{:.2%}', 'DivYieldMedian': '{:.2%}'})


In [ ]:
# Calculate M&M's

mean20 = dp_df1['FwdDiv%'].mean().tolist()
median20 = dp_df1['FwdDiv%'].median().tolist()

dp_l10_df = dp_df1[dp_df1['Date'].dt.year >= cutoff10]
mean10 = dp_l10_df['FwdDiv%'].mean().tolist()
median10 = dp_l10_df['FwdDiv%'].median().tolist()

dp_l5_df = dp_df1[dp_df1['Date'].dt.year >= cutoff5]
mean5 = dp_l5_df['FwdDiv%'].mean().tolist()
median5 = dp_l5_df['FwdDiv%'].median().tolist()

dp_l3_df = dp_df1[dp_df1['Date'].dt.year >= cutoff3]
mean3 = dp_l3_df['FwdDiv%'].mean().tolist()
median3 = dp_l3_df['FwdDiv%'].median().tolist()

print(f'Mean20Yr Div: {round(mean20 * 100, 2)}%')
print(f'Median20Yr Div: {round(median20 * 100, 2)}%')
print()
print(f'Mean10Yr Div: {round(mean10 * 100, 2)}%')
print(f'Median10Yr Div: {round(median10 * 100, 2)}%')
print()
print(f'Mean5Yr Div: {round(mean5 * 100, 2)}%')
print(f'Median5Yr Div: {round(median5 * 100, 2)}%')
print()
print(f'Mean3Yr Div: {round(mean3 * 100, 2)}%')
print(f'Median3Yr Div: {round(median3 * 100, 2)}%')
print()

In [ ]:
dyt_gate = .0300
dyt_gate10 = round((dyt_gate * .1) + dyt_gate, 4)
dyt_gate20 = round((dyt_gate * .20) + dyt_gate, 4)

print(f'The DivBond Gate is: {round(dyt_gate * 100, 2)}%')
print(f'The DivBond Gate10 is: {round(dyt_gate10 * 100, 2)}%')
print(f'The DivBond Gate20 is: {round(dyt_gate20 * 100, 2)}%')

In [ ]:
dyt_fig = go.Figure([
    go.Scatter(name='DYT', y=round(dp_df1['FwdDivYield'] * 100, 2), x=dp_df1['Date'], mode='lines', marker_color='Blue')
])
dyt_fig.add_hline(y=round(dyt_gate * 100, 2), line_dash="dash", line_color="red", annotation_text="Gate", annotation_position="right")
dyt_fig.add_hline(y=round(dyt_gate10 * 100, 2), line_dash="dash", line_color="yellow", annotation_text="Gate10", annotation_position="right")
dyt_fig.add_hline(y=round(dyt_gate20 * 100, 2), line_dash="dash", line_color="green", annotation_text="Gate20", annotation_position="right")
dyt_fig.update_layout(yaxis_title='FwdDivYield %', xaxis_title='FiscalYear', title='DYT', template='plotly_dark')
dyt_fig.show()

## Margins
Steps
1. Profit Margins
2. Cash Margins
3. Dividend Margins

In [ ]:
gpm_median20 = margin_df['GPM'].tail(20).median().tolist()
gpm_median10 = margin_df['GPM'].tail(10).median().tolist()
gpm_median5 = margin_df['GPM'].tail(5).median().tolist()
gpm_median3 = margin_df['GPM'].tail(3).median().tolist()

opm_median20 = margin_df['OPM'].tail(20).median().tolist()
opm_median10 = margin_df['OPM'].tail(10).median().tolist()
opm_median5 = margin_df['OPM'].tail(5).median().tolist()
opm_median3 = margin_df['OPM'].tail(3).median().tolist()


npm_median20 = margin_df['NPM'].tail(20).median().tolist()
npm_median10 = margin_df['NPM'].tail(10).median().tolist()
npm_median5 = margin_df['NPM'].tail(5).median().tolist()
npm_median3 = margin_df['NPM'].tail(3).median().tolist()



In [ ]:
profit_fig1 = go.Figure(data=[
    go.Bar(name='GPM', x=margin_df['FiscalYear'], y=round(margin_df['GPM'] * 100, 2), offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='OPM', x=margin_df['FiscalYear'], y=round(margin_df['OPM'] * 100, 2), offsetgroup=2, marker_color='Blue'),
    go.Bar(name='NPM', x=margin_df['FiscalYear'], y=round(margin_df['NPM'] * 100, 2), offsetgroup=3, marker_color='LightBlue'),

])
profit_fig1.update_xaxes(dtick=1)
profit_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Profit Margins', template='plotly_dark')
profit_fig1.show()

In [ ]:

print(f'Median20Year GPM: {round(gpm_median20 * 100, 2)}%')
print(f'Median10Year GPM: {round(gpm_median10 * 100, 2)}%')
print(f'Median5Year GPM: {round(gpm_median5 * 100, 2)}%')
print(f'Median3Year GPM: {round(gpm_median3 * 100, 2)}%')
print()

print(f'Median20Year OPM: {round(opm_median20 * 100, 2)}%')
print(f'Median10Year OPM: {round(opm_median10 * 100, 2)}%')
print(f'Median5Year OPM: {round(opm_median5 * 100, 2)}%')
print(f'Median3Year OPM: {round(opm_median3 * 100, 2)}%')
print()

print(f'Median20Year NPM: {round(npm_median20 * 100, 2)}%')
print(f'Median10Year NPM: {round(npm_median10 * 100, 2)}%')
print(f'Median5Year NPM: {round(npm_median5 * 100, 2)}%')
print(f'Median3Year NPM: {round(npm_median3 * 100, 2)}%')


In [ ]:
gpm_gate = .60
opm_gate = .35
npm_gate = .30

In [ ]:
ocm_median20 = margin_df['OCM'].tail(20).median().tolist()
ocm_median10 = margin_df['OCM'].tail(10).median().tolist()
ocm_median5 = margin_df['OCM'].tail(5).median().tolist()
ocm_median3 = margin_df['OCM'].tail(3).median().tolist()

fcm_median20 = margin_df['FCM'].tail(20).median().tolist()
fcm_median10 = margin_df['FCM'].tail(10).median().tolist()
fcm_median5 = margin_df['FCM'].tail(5).median().tolist()
fcm_median3 = margin_df['FCM'].tail(3).median().tolist()


cap_median20 = margin_df['CXM'].tail(20).median()
cap_median10 = margin_df['CXM'].tail(10).median()
cap_median5 = margin_df['CXM'].tail(5).median()
cap_median3 = margin_df['CXM'].tail(3).median()

In [ ]:
cash_fig1 = go.Figure(data=[
    go.Bar(name='OCM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['OCM'].tail(20) * 100, 2), offsetgroup=1, marker_color='Green'),
    go.Bar(name='FCM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['FCM'].tail(20) * 100, 2), offsetgroup=2, marker_color='LightGreen'),
    go.Bar(name='CXM', x=margin_df['FiscalYear'].tail(20), y=round(margin_df['CXM'].tail(20) * 100, 2).abs(), offsetgroup=3, marker_color='Red'),

])
cash_fig1.update_xaxes(dtick=1)
cash_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Cash Margins', template='plotly_dark')
cash_fig1.show()

In [ ]:
print(f'Median20Year OCM: {round(ocm_median20 * 100, 2)}%')
print(f'Median10Year OCM: {round(ocm_median10 * 100, 2)}%')
print(f'Median5Year OCM: {round(ocm_median5 * 100, 2)}%')
print(f'Median3Year OCM: {round(ocm_median3 * 100, 2)}%')
print()
print(f'Median20Year FCM: {round(fcm_median20 * 100, 2)}%')
print(f'Median10Year FCM: {round(fcm_median10 * 100, 2)}%')
print(f'Median5Year FCM: {round(fcm_median5 * 100, 2)}%')
print(f'Median3Year FCM: {round(fcm_median3 * 100, 2)}%')
print()
print(f'Median20Year CXM: {round(cap_median20 * 100, 2)}%')
print(f'Median10Year CXM: {round(cap_median10 * 100, 2)}%')
print(f'Median5Year CXM: {round(cap_median5 * 100, 2)}%')
print(f'Median3Year CXM: {round(cap_median3 * 100, 2)}%')

In [ ]:
ocm_gate = .40
fcm_gate = .25
cxm_gate = .15

## DVD - Dividend Vs Debt
Steps
1. Dividend Covered by OpCash
2. Dividend Covered by FreeCash
3. Total Cash On Hand vs Total Debts & CAPEX
4. Operating Cash vs Current Liabilities & CAPEX

In [ ]:
dop_median20 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(20).median().tolist()
dop_median10 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(10).median().tolist()
dop_median5 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(5).median().tolist()
dop_median3 = (dvd_df['DivCash']/dvd_df['OpCash']).tail(3).median().tolist()

dfr_median20 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(20).median().tolist()
dfr_median10 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(10).median().tolist()
dfr_median5 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(5).median().tolist()
dfr_median3 = (dvd_df['DivCash']/dvd_df['FreeCash']).tail(3).median().tolist()

In [ ]:
div_fig1 = go.Figure(data=[
    go.Bar(name='DOCM', x=dvd_df['FiscalYear'], y=round((dvd_df['DivCash']/dvd_df['OpCash']).abs() * 100, 2), offsetgroup=1, marker_color='Green'),
    go.Bar(name='DFCM', x=dvd_df['FiscalYear'], y=round((dvd_df['DivCash']/dvd_df['FreeCash']).abs() * 100, 2), offsetgroup=2, marker_color='LightGreen')
])
div_fig1.update_xaxes(dtick=1)
div_fig1.update_yaxes(range=[0,100])
div_fig1.update_layout(yaxis_title='Margin %', xaxis_title='FiscalYear', title='Div Margins ', template='plotly_dark')
div_fig1.show()

In [ ]:
print(f'Median20Year Div Op Cash Margin: {abs(round(dop_median20 * 100, 2))}%')
print(f'Median10Year Div Op Cash Margin: {abs(round(dop_median10 * 100, 2))}%')
print(f'Median5Year Div Op Cash Margin: {abs(round(dop_median5 * 100, 2))}%')
print(f'Median3Year Div Op Cash Margin: {abs(round(dop_median3 * 100, 2))}%')
print()
print(f'Median20Year Div Free Cash Margin: {abs(round(dfr_median20 * 100, 2))}%')
print(f'Median10Year Div Free Cash Margin: {abs(round(dfr_median10 * 100, 2))}%')
print(f'Median5Year Div Free Cash Margin: {abs(round(dfr_median5 * 100, 2))}%')
print(f'Median3Year Div Free Cash Margin: {abs(round(dfr_median3 * 100, 2))}%')

In [ ]:
docm_gate = .70
dfcm_gate = .90

In [ ]:
div_fig2 = go.Figure(data=[
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=1, marker_color='DarkBlue'),
    go.Bar(name='FreeCash', x=dvd_df['FiscalYear'], y=dvd_df['FreeCash'], offsetgroup=2, marker_color='Blue'),
    go.Bar(name='Div', x=dvd_df['FiscalYear'], y=dvd_df['DivCash'].abs(), offsetgroup=3, marker_color='DarkGreen'),
    go.Bar(name='BuyBack', x=dvd_df['FiscalYear'], y=(dvd_df['StockIssue'] + dvd_df['StockBuyBack']).abs(), offsetgroup=3, marker_color='Yellow',
            base=dvd_df['DivCash'].abs())
])
div_fig2.update_layout(barmode='group')
div_fig2.update_xaxes(dtick=1)
div_fig2.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Div Coverage ', template='plotly_dark')
div_fig2.show()

In [ ]:
dvd_div10 = dvd_df['DivCash'].tail(10).sum().tolist()
dvd_bb10 = abs(dvd_df['StockBuyBack'] + dvd_df['StockIssue']).tail(10).sum().tolist()
dvd_opcash10 = dvd_df['OpCash'].tail(10).sum().tolist()

print(f'Year10 Dividend Return: $ {abs(dvd_div10)}')
print(f'Year10 Stock Buy Back Return: $ {abs(dvd_bb10)}')
print(f'Year10 % of Op Cash Return Via Div: {abs(round((dvd_div10 / dvd_opcash10) * 100, 2))}%')
print(f'Year10 % of Op Cash Return Via BB: {abs(round((dvd_bb10 / dvd_opcash10) * 100, 2))}%')

In [ ]:
debt_fig1 = go.Figure(data=[
    go.Bar(name='Cash', x=dvd_df['FiscalYear'], y=abs(dvd_df['C&E']), offsetgroup=2, marker_color='DarkBlue'),
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=2, marker_color='Blue', base=dvd_df['C&E']),
    go.Bar(name='CurrentDebt', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=3, marker_color='DarkRed'),
    go.Bar(name='TotalDebt', x=dvd_df['FiscalYear'], y=(dvd_df['TotalLiabilities'] - dvd_df['CurrentLiabilities']), offsetgroup=3,
           marker_color='Red', base=dvd_df['CurrentLiabilities']),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=4, marker_color='Yellow'),
])
debt_fig1.update_layout(barmode='group')
debt_fig1.update_xaxes(dtick=1)
debt_fig1.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Debt Coverage By Cash', template='plotly_dark')
debt_fig1.show()

In [ ]:
debt_fig2 = go.Figure(data=[
    go.Bar(name='TreasuryStock', x=dvd_df['FiscalYear'], y=abs(dvd_df['TreasuryStock']), offsetgroup=1, marker_color='Green'),
    go.Bar(name='Cash', x=dvd_df['FiscalYear'], y=abs(dvd_df['C&E']), offsetgroup=2, marker_color='DarkBlue'),
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=2, marker_color='Blue', base=dvd_df['C&E']),
    go.Bar(name='CurrentLiabilities', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=3, marker_color='DarkRed'),
    go.Bar(name='TotalDebt', x=dvd_df['FiscalYear'], y=(dvd_df['TotalLiabilities'] - dvd_df['CurrentLiabilities']), offsetgroup=3,
           marker_color='Red', base=dvd_df['CurrentLiabilities']),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=4, marker_color='Yellow'),
])
debt_fig2.update_layout(barmode='group')
debt_fig2.update_xaxes(dtick=1)
debt_fig2.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Debt Coverage By Cash & Treasury', template='plotly_dark')
debt_fig2.show()

In [ ]:
debt_fig3 = go.Figure(data=[
    go.Bar(name='OpCash', x=dvd_df['FiscalYear'], y=dvd_df['OpCash'], offsetgroup=1, marker_color='Green'),
    go.Bar(name='CurrentLiabilities', x=dvd_df['FiscalYear'], y=(dvd_df['CurrentLiabilities']), offsetgroup=2, marker_color='Red'),
    go.Bar(name='Capex', x=dvd_df['FiscalYear'], y=(abs(dvd_df['CAPEX'])), offsetgroup=3, marker_color='Yellow'),
])
debt_fig3.update_layout(barmode='group')
debt_fig3.update_xaxes(dtick=1)
debt_fig3.update_layout(yaxis_title='Million', xaxis_title='FiscalYear', title='Current Liabilities Coverage', template='plotly_dark')
debt_fig3.show()

## Growth
- Revenue Growth
- Operating Cash Growth
- Dividend Growth

In [ ]:
gro_df1[['FiscalYear', 'Revenue', 'RevGro','RevPS', 'RevGroPS']]

In [ ]:
rev_bottom_3 = gro_df1[['Revenue']].head(3).mean().tolist()
rev_top_3 = gro_df1[['Revenue']].tail(3).mean().tolist()
cagr_rev = ((rev_top_3[0]/rev_bottom_3[0]) ** (1/20)) - 1


rps_bottom_3 = gro_df1[['RevPS']].head(3).mean().tolist()
rps_top_3 = gro_df1[['RevPS']].tail(3).mean().tolist()
cagr_rps = ((rps_top_3[0]/rps_bottom_3[0]) ** (1/20)) - 1

print(f'The CAGR of Total Company Revenue is: {round(cagr_rev * 100,2 )}%')
print(f'The CAGR of Revenue Per Share is: {round(cagr_rps * 100,2 )}%')

In [ ]:
gro_df1[['FiscalYear', 'OpCash', 'OpCashGro','OpCashPS', 'OpCashGroPS']]

In [ ]:
oc_bottom_3 = gro_df1[['OpCash']].head(3).mean().tolist()
oc_top_3 = gro_df1[['OpCash']].tail(3).mean().tolist()
cagr_opcash = ((oc_top_3[0]/oc_bottom_3[0]) ** (1/20)) - 1


ocps_bottom_3 = gro_df1[['OpCashPS']].head(3).mean().tolist()
ocps_top_3 = gro_df1[['OpCashPS']].tail(3).mean().tolist()
cagr_opcash_ps = ((ocps_top_3[0]/ocps_bottom_3[0]) ** (1/20)) - 1

print(f'The CAGR of Total Company Operating Cash is: {round(cagr_opcash * 100,2 )}%')
print(f'The CAGR of Operating Cash Per Share is: {round(cagr_opcash_ps * 100,2 )}%')

In [ ]:
gro_df1[['FiscalYear', 'DivCash', 'DivGro','DivPS', 'DivGroPS']]

In [ ]:
divgro20 = gro_df1[['DivGro']].tail(20).median().tolist()
divgro10 = gro_df1[['DivGro']].tail(10).median().tolist()
divgro5 = gro_df1[['DivGro']].tail(5).median().tolist()
divgro3 = gro_df1[['DivGro']].tail(3).median().tolist()


dpsgro20 = gro_df1[['DivGroPS']].tail(20).median().tolist()
dpsgro10 = gro_df1[['DivGroPS']].tail(10).median().tolist()
dpsgro5 = gro_df1[['DivGroPS']].tail(5).median().tolist()
dpsgro3 = gro_df1[['DivGroPS']].tail(3).median().tolist()

print(f'Year20 DivGro: {round(divgro20[0] * 100,2 )}% ')
print(f'Year10 DivGro: {round(divgro10[0] * 100,2 )}% ')
print(f'Year5 DivGro: {round(divgro5[0] * 100,2 )}% ')
print(f'Year3 DivGro: {round(divgro3[0] * 100,2 )}% ')
print()
print(f'Year20 DpsGro: {round(dpsgro20[0] * 100,2 )}% ')
print(f'Year10 DpsGro: {round(dpsgro10[0] * 100,2 )}% ')
print(f'Year5 DpsGro: {round(dpsgro5[0] * 100,2 )}% ')
print(f'Year3 DpsGro: {round(dpsgro3[0] * 100,2 )}% ')

## Notebook End Audit Ouput

In [ ]:
audit_json = {
    "analysis_type": "div-bond",
    "analysis_date": today.strftime('%Y-%m-%d'),
    "last_df_date":dp_df1['Date'].iloc[-1].strftime('%Y-%m-%d'),
    "mean20yr_div_yield": mean20,
    "median20yr_div_yield": median20,
    "mean10yr_div_yield": mean10,
    "median10yr_div_yield": median10,
    "mean5yr_div_yield": median5,
    "median5yr_div_yield": median5,
    "mean3yr_div_yield": median3,
    "median3yr_div_yield": median3,
    "dyt_gate": dyt_gate,
    "dyt_gate10": dyt_gate10,
    "dyt_gate20": dyt_gate20,
    "gpm_gate": gpm_gate,
    "opm_gate": opm_gate,
    "npm_gate": npm_gate,
    "ocm_gate": ocm_gate,
    "fcm_gate": fcm_gate,
    "cxm_gate": cxm_gate,
    "div_opcash_gate": docm_gate,
    "div_freecash_gate": dfcm_gate,
    "year10_div_return_total": dvd_div10,
    "year10_bb_return_total": dvd_bb10,
    "year10_opcash_total": dvd_opcash10,
    "cagr_rev": cagr_rev,
    "cagr_rps": cagr_rps,
    "cagr_opcash": cagr_opcash,
    "cagr_opcash_ps": cagr_opcash_ps,
    "divgro20_median": divgro20[0],
    "divgro10_median": divgro10[0],
    "divgro5_median": divgro5[0],
    "divgro3_median": divgro3[0],
    "dpsgro20_median": dpsgro20[0],
    "dpsgro10_median": dpsgro10[0],
    "dpsgro5_median": dpsgro5[0],
    "dpsgro3_median": dpsgro3[0]
}

audit_json

In [ ]:
audit_df = pd.DataFrame([audit_json])
audit_df

if audit == True:
    if os.path.isfile(path_analysis_csv) :
        audit_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
    else:
        audit_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)
else:
    print(f'Audit = {audit}')